# Phase 9: EXP-12B YOLO11l Model Scaling

This notebook trains YOLO11l on the existing single-class dataset to determine if scaling up the architecture improves Stage-1 recall over the YOLO11m baseline.

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Install dependencies
!pip install ultralytics

In [3]:
import os
import sys
import torch

PROJECT_ROOT = '/content/drive/MyDrive/sem_defect_project'
os.chdir(PROJECT_ROOT)
print(f"Current working directory: {os.getcwd()}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Current working directory: /content/drive/MyDrive/sem_defect_project
CUDA available: True
GPU: Tesla T4


In [4]:
# We will use the data directly from Google Drive.
data_yaml_path = 'dataset_yolo_single_class/data.yaml'

print(f"Using dataset configuration from: {data_yaml_path}")

Using dataset configuration from: dataset_yolo_single_class/data.yaml


## EXP-12B: YOLO11l Training (200 Epochs)

In [ ]:
from ultralytics import YOLO

# Initialize YOLO11l
model_l = YOLO('yolo11l.pt')

# Add a callback to clearly print the epoch number to avoid Colab truncation hiding progress
def print_epoch(trainer):
    print(f"\n>>> Completed Epoch {trainer.epoch + 1} / {trainer.epochs} <<<\n", flush=True)

model_l.add_callback("on_train_epoch_end", print_epoch)

results_l = model_l.train(
    data=data_yaml_path,
    epochs=200,
    patience=30, # Stop after 30 epochs with no validation improvement
    save=True,   # Ensure best.pt is saved
    imgsz=640,
    batch=16,
    project='runs/detect',
    name='EXP-12B_YOLO11l',
    seed=42
)


Ultralytics 8.4.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_yolo_single_class/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=EXP-12B_YOLO11l-5, nbs=64, nms=None, ops

## Validation Metrics Extraction
Extract and print the precise validation metrics for comparison.

In [ ]:
# Validate the model using best weights
val_results = model_l.val(
    data=data_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    conf=0.15,
    iou=0.50
)

print('\n--- EXP-12B Validation Results ---')
p = val_results.results_dict.get('metrics/precision(B)', 0)
r = val_results.results_dict.get('metrics/recall(B)', 0)
f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0

print(f"Precision: {p:.4f}")
print(f"Recall: {r:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"mAP50: {val_results.results_dict.get('metrics/mAP50(B)', 0):.4f}")
print(f"mAP50-95: {val_results.results_dict.get('metrics/mAP50-95(B)', 0):.4f}")
print(f"Training parameters: {model_l.info()}")


: 